# TakeMeter — Discourse Classifier Fine-Tuning

**Runtime → Change runtime type → T4 GPU** before running.

Workflow:
1. Install deps & check GPU
2. Upload `dataset.csv`
3. Log in to Hugging Face
4. Run 5-fold CV on DeBERTa-v3-base (expected front-runner)
5. Optionally run the full 4-model bake-off
6. Retrain best config on all data → push to Hub


## 1. Check GPU

In [ ]:
!nvidia-smi
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Install dependencies

In [ ]:
!pip install -q transformers datasets accelerate evaluate scikit-learn sentencepiece protobuf

## 3. Upload dataset

Run the cell below and select your local `dataset.csv`.

In [ ]:
from google.colab import files
uploaded = files.upload()  # select dataset.csv from your machine

import pandas as pd
df_check = pd.read_csv("dataset.csv")
print(f"Rows: {len(df_check)}")
print(df_check["label"].value_counts())

## 4. Log in to Hugging Face

You need a write-access token: huggingface.co → Settings → Access Tokens → New token (write).

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 5. Config & imports

In [ ]:
import json
import os
import random
import warnings

import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight

import torch
import torchvision.io as _tvio
if not hasattr(_tvio, "VideoReader"):
    class _VideoReaderStub: pass
    _tvio.VideoReader = _VideoReaderStub

from datasets import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

warnings.filterwarnings("ignore", category=FutureWarning)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# ── Labels ────────────────────────────────────────────────────────────────────
LABEL2ID = {"Analytical": 0, "Evaluative": 1, "Informational": 2, "Reactive": 3}
ID2LABEL  = {v: k for k, v in LABEL2ID.items()}
NUM_LABELS = len(LABEL2ID)

# ── Model candidates ──────────────────────────────────────────────────────────
MODELS = {
    "deberta-v3-base":  "microsoft/deberta-v3-base",
    "ModernBERT-base":  "answerdotai/ModernBERT-base",
    "roberta-base":     "roberta-base",
    "distilbert-base":  "distilbert-base-uncased",
}

# ── Hyperparameters ───────────────────────────────────────────────────────────
LR_GRID    = [1e-5, 2e-5, 3e-5]
N_FOLDS    = 5
SEED       = 42
MAX_LEN    = 256
BATCH_SIZE = 16
MAX_EPOCHS = 10
PATIENCE   = 3

print("Config ready.")

## 6. Data loading & hold-out split

In [ ]:
from sklearn.model_selection import train_test_split

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def build_input_text(row):
    tag = "[POST]" if row["type"] == "post" else "[COMMENT]"
    return f"{tag} {row['post_title']}\n{row['text']}"

set_seed(SEED)

df = pd.read_csv("dataset.csv")
df = df[df["label"].notna() & (df["label"].str.strip() != "")].copy()
df["input_text"] = df.apply(build_input_text, axis=1)
df["label_id"]   = df["label"].map(LABEL2ID)
df = df.reset_index(drop=True)

# Set aside ~40 stratified examples as the final hold-out (touched once, at the very end)
cv_df, holdout_df = train_test_split(
    df, test_size=40, stratify=df["label_id"], random_state=SEED
)
cv_df      = cv_df.reset_index(drop=True)
holdout_df = holdout_df.reset_index(drop=True)

print(f"CV pool:      {len(cv_df)} examples")
print(f"Hold-out set: {len(holdout_df)} examples  ← do not touch until final eval")
print("\nCV pool label distribution:")
print(cv_df["label"].value_counts())

## 7. Training utilities

In [ ]:
def make_hf_dataset(texts, label_ids, tokenizer):
    ds = Dataset.from_dict({"input_text": texts, "labels": label_ids})
    def tokenize(batch):
        return tokenizer(batch["input_text"], truncation=True,
                         max_length=MAX_LEN, padding="max_length")
    ds = ds.map(tokenize, batched=True, remove_columns=["input_text"])
    ds.set_format("torch")
    return ds


class WeightedTrainer(Trainer):
    def __init__(self, class_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = torch.nn.CrossEntropyLoss(
            weight=self.class_weights.to(outputs.logits.device)
        )(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"macro_f1": f1_score(labels, preds, average="macro", zero_division=0)}


def train_fold(model_hf_id, lr, fold_idx, train_df, val_df, class_weights):
    tokenizer = AutoTokenizer.from_pretrained(model_hf_id)
    train_ds  = make_hf_dataset(train_df["input_text"].tolist(), train_df["label_id"].tolist(), tokenizer)
    val_ds    = make_hf_dataset(val_df["input_text"].tolist(),   val_df["label_id"].tolist(),   tokenizer)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_hf_id, num_labels=NUM_LABELS,
        id2label=ID2LABEL, label2id=LABEL2ID, ignore_mismatched_sizes=True,
    )

    warmup_steps = max(1, int(0.1 * (len(train_ds) // BATCH_SIZE) * MAX_EPOCHS))

    training_args = TrainingArguments(
        output_dir=f"/content/ckpts/fold{fold_idx}",
        num_train_epochs=MAX_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        learning_rate=lr,
        warmup_steps=warmup_steps,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        seed=SEED,
        report_to="none",
        logging_steps=50,
        fp16=False,  # DeBERTa-v3 is incompatible with fp16
    )

    trainer = WeightedTrainer(
        class_weights=class_weights,
        model=model, args=training_args,
        train_dataset=train_ds, eval_dataset=val_ds,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)],
    )
    trainer.train()

    pred_out = trainer.predict(val_ds)
    preds    = np.argmax(pred_out.predictions, axis=-1)
    true     = pred_out.label_ids
    macro_f1 = f1_score(true, preds, average="macro", zero_division=0)

    print(f"  Fold {fold_idx+1} macro-F1: {macro_f1:.4f}")
    print(classification_report(true, preds, target_names=list(LABEL2ID.keys()), zero_division=0))

    per_class = f1_score(true, preds, average=None, zero_division=0)
    return {
        "fold": fold_idx,
        "macro_f1": macro_f1,
        "per_class_f1": {k: float(per_class[v]) for k, v in LABEL2ID.items()},
        "confusion_matrix": confusion_matrix(true, preds).tolist(),
    }


def run_cv(model_name, model_hf_id, lr, data_df):
    set_seed(SEED)
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    class_weights = torch.tensor(
        compute_class_weight("balanced", classes=np.arange(NUM_LABELS), y=data_df["label_id"].values),
        dtype=torch.float32,
    )

    fold_results = []
    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(data_df, data_df["label_id"])):
        print(f"\n{'='*60}")
        print(f"  {model_name}  |  lr={lr:.0e}  |  fold {fold_idx+1}/{N_FOLDS}")
        print(f"{'='*60}")
        result = train_fold(
            model_hf_id, lr, fold_idx,
            data_df.iloc[train_idx].reset_index(drop=True),
            data_df.iloc[val_idx].reset_index(drop=True),
            class_weights,
        )
        fold_results.append(result)

    macro_f1s = [r["macro_f1"] for r in fold_results]
    mean, std = float(np.mean(macro_f1s)), float(np.std(macro_f1s))
    print(f"\n>>> {model_name}  lr={lr:.0e}  macro-F1 = {mean:.4f} ± {std:.4f}")
    return {"model": model_name, "lr": lr, "mean_macro_f1": mean, "std_macro_f1": std, "folds": fold_results}


print("Utilities ready.")

## 8. CV — DeBERTa-v3-base (recommended starting point)

Runs 3 learning rates × 5 folds = 15 training runs (~30–45 min on T4).

In [ ]:
deberta_results = []

for lr in LR_GRID:
    result = run_cv("deberta-v3-base", MODELS["deberta-v3-base"], lr, cv_df)
    deberta_results.append(result)

    # Save incrementally
    with open("cv_results.json", "w") as f:
        json.dump(deberta_results, f, indent=2)

print("\n── DeBERTa-v3-base results ──")
for r in sorted(deberta_results, key=lambda x: x["mean_macro_f1"], reverse=True):
    print(f"  lr={r['lr']:.0e}  macro-F1 = {r['mean_macro_f1']:.4f} ± {r['std_macro_f1']:.4f}")

## 9. (Optional) Full bake-off — all 4 models

Skip this if DeBERTa already clears macro-F1 ≥ 0.80. Run it if you want to confirm no other model beats it (~2 hrs total).

In [ ]:
all_results = list(deberta_results)  # start from DeBERTa results already computed

for model_name, model_hf_id in MODELS.items():
    if model_name == "deberta-v3-base":
        continue  # already done
    for lr in LR_GRID:
        result = run_cv(model_name, model_hf_id, lr, cv_df)
        all_results.append(result)
        with open("cv_results_full.json", "w") as f:
            json.dump(all_results, f, indent=2)

print("\n── Full bake-off leaderboard ──")
for r in sorted(all_results, key=lambda x: x["mean_macro_f1"], reverse=True):
    print(f"  {r['model']:30s}  lr={r['lr']:.0e}  macro-F1 = {r['mean_macro_f1']:.4f} ± {r['std_macro_f1']:.4f}")

## 10. Pick the best config

Edit `BEST_MODEL` and `BEST_LR` below to match the winner from your CV results.

In [ ]:
# ── Set these to the winning model + LR from your CV results ──────────────────
BEST_MODEL = "deberta-v3-base"  # change if another model won
BEST_LR    = 2e-5               # change to whichever LR had the highest mean macro-F1
HF_REPO    = "your-username/takemeter"  # ← replace with your HF username

print(f"Best config: {BEST_MODEL}, lr={BEST_LR:.0e}")
print(f"Will push to: {HF_REPO}")

## 11. Final training — retrain on all CV data

Uses the winning config. The 40-example hold-out is evaluated **once** here to get an unbiased final score.

In [ ]:
set_seed(SEED)

model_hf_id = MODELS[BEST_MODEL]
tokenizer   = AutoTokenizer.from_pretrained(model_hf_id)

class_weights = torch.tensor(
    compute_class_weight("balanced", classes=np.arange(NUM_LABELS), y=cv_df["label_id"].values),
    dtype=torch.float32,
)

train_ds   = make_hf_dataset(cv_df["input_text"].tolist(),      cv_df["label_id"].tolist(),      tokenizer)
holdout_ds = make_hf_dataset(holdout_df["input_text"].tolist(), holdout_df["label_id"].tolist(), tokenizer)

model = AutoModelForSequenceClassification.from_pretrained(
    model_hf_id, num_labels=NUM_LABELS,
    id2label=ID2LABEL, label2id=LABEL2ID, ignore_mismatched_sizes=True,
)

warmup_steps = max(1, int(0.1 * (len(train_ds) // BATCH_SIZE) * MAX_EPOCHS))

final_args = TrainingArguments(
    output_dir="/content/takemeter-final",
    num_train_epochs=MAX_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=BEST_LR,
    warmup_steps=warmup_steps,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    seed=SEED,
    report_to="none",
    logging_steps=50,
    fp16=False,  # DeBERTa-v3 is incompatible with fp16
    push_to_hub=True,
    hub_model_id=HF_REPO,
)

final_trainer = WeightedTrainer(
    class_weights=class_weights,
    model=model, args=final_args,
    train_dataset=train_ds, eval_dataset=holdout_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)],
    tokenizer=tokenizer,
)

final_trainer.train()
print("Training complete.")

## 12. Final evaluation on hold-out (run once)

In [ ]:
pred_out     = final_trainer.predict(holdout_ds)
final_preds  = np.argmax(pred_out.predictions, axis=-1)
final_true   = pred_out.label_ids
final_macro  = f1_score(final_true, final_preds, average="macro", zero_division=0)

print(f"Hold-out macro-F1: {final_macro:.4f}")
print()
print(classification_report(final_true, final_preds, target_names=list(LABEL2ID.keys()), zero_division=0))
print("Confusion matrix (rows=true, cols=pred):")
cm = confusion_matrix(final_true, final_preds)
cm_df = pd.DataFrame(cm, index=list(LABEL2ID.keys()), columns=list(LABEL2ID.keys()))
print(cm_df)

## 13. Push to Hugging Face Hub

In [ ]:
# Load CV summary to include in the model card
try:
    with open("cv_results.json") as f:
        cv_data = json.load(f)
    best_cv = next(r for r in cv_data if r["lr"] == BEST_LR)
    cv_summary = f"{best_cv['mean_macro_f1']:.4f} ± {best_cv['std_macro_f1']:.4f}"
except Exception:
    cv_summary = "see cv_results.json"

model_card = f"""---
language: en
tags:
- text-classification
- anime
- discourse
pipeline_tag: text-classification
---

# TakeMeter — Anime Discourse Classifier

Fine-tuned **{BEST_MODEL}** for 4-class discourse mode classification of anime community posts.

## Labels
| ID | Label | Description |
|---|---|---|
| 0 | Analytical | Argues an interpretation with text evidence |
| 1 | Evaluative | Asserts a judgment or ranking |
| 2 | Informational | Seeks or supplies factual/lore information |
| 3 | Reactive | Expresses emotion or hype, no claim |

## Training
- **Data:** 812 labeled posts/comments from r/attackontitan, r/HunterXHunter, r/FullmetalAlchemist, r/evangelion
- **CV macro-F1:** {cv_summary} (5-fold stratified)
- **Hold-out macro-F1:** {final_macro:.4f}
- **Class weighting:** inverse-frequency to handle label imbalance

## Usage
```python
from transformers import pipeline
classifier = pipeline(\"text-classification\", model=\"{HF_REPO}\")
classifier(\"[POST] Was the ending good?\\nI think Eren's arc was the most compelling...\")  
# → {{\"label\": \"Analytical\", \"score\": 0.87}}
```
"""

with open("/content/takemeter-final/README.md", "w") as f:
    f.write(model_card)

final_trainer.push_to_hub(commit_message=f"Final model — hold-out macro-F1 {final_macro:.4f}")
print(f"Pushed to https://huggingface.co/{HF_REPO}")

## 14. Download results

Save `cv_results.json` to your local machine before the Colab session ends.

In [ ]:
from google.colab import files
files.download("cv_results.json")